In [1]:

print("ok")

ok


In [6]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter



c:\Users\VENKATESH\anaconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def load_pdf_file(data):
    loader=DirectoryLoader(data,glob='*.pdf',loader_cls=PyPDFLoader)
    documents=loader.load()
    return documents

In [8]:
extracted_data=load_pdf_file(data='Data/')

In [9]:
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [10]:
text_chunks=text_split(extracted_data)
print(len(text_chunks))

5860


In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [12]:
def download_hugging_face():
    embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [13]:


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

query_result = embeddings.embed_query("Hello world")
print(len(query_result))

C:\Users\VENKATESH\AppData\Local\Temp\ipykernel_22508\2378937441.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2350.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


384


In [14]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')

In [15]:
from dotenv import load_dotenv
import os

load_dotenv()



True

In [16]:
import os

os.environ["PINECONE_API_KEY"] = "pcsk_2u5baP_KxS8L6nqv1W44Z26pSRsWaoZE3sYcqMq54BDKQ1M6qnqRHbYHaih1YFr4zwb8Y3"

In [ ]:
from pinecone import Pinecone, ServerlessSpec
import os

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medicalbot"

# Check existing indexes
existing_indexes = [index.name for index in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("✅ Index created successfully!")
else:
    print("⚡ Index already exists!")

✅ Index created successfully!


In [18]:
from pinecone import Pinecone


In [19]:
import os
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')

In [20]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings,
)

In [21]:
from langchain_pinecone import PineconeVectorStore

In [22]:
docsearch=PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
    )

In [23]:
docsearch

In [24]:
retriever=docsearch.as_retriever(search_type='similarity',search_kwargs={'k':3})
retrieved_docs=retriever.invoke("what is acne")


In [25]:
retrieved_docs

[Document(id='71fec286-89b3-4cf9-946d-f920748993de', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39.0, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Gale Encyclopedia of Medicine Vol. 1 (A-B).pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='5792bb66-3f23-4e4a-b407-7ef5ea441111', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 37.0, 'page_label': '38', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Gale Encyclopedia of Medicine Vol. 1 (A-B).pdf', 'total_pages': 637.0}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the

In [32]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LLM
llm = OllamaLLM(model="llama3.1:8b")  # use 8b (stable), not 18b for now

# Prompt
system_prompt = """
You are a helpful medical assistant.
Use the given retrieved context to answer the question.
If not found, say you don't know.
Keep answer within 3 sentences.

Context:
{context}

Question:
{question}
"""

prompt = ChatPromptTemplate.from_template(system_prompt)

# Limit retrieval
retriever = retriever.with_config(search_kwargs={"k": 3})

# Format docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG Chain (NEW STYLE)
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": lambda x: x
    }
    | prompt
    | llm
    | StrOutputParser()
)



In [38]:
# Run query
response = rag_chain.invoke("what is cancer and how we remove them?")
print(response)

Cancer refers to the uncontrolled growth and spread of abnormal cells in the body, specifically the malignant type mentioned in the context. Treatment for cancerous tumors often involves a combination of surgery, chemotherapy, and radiation therapy. In some cases, as mentioned in the lumpectomy procedure, only the tumor and surrounding normal tissue are removed surgically to treat breast cancer.


In [39]:
docs = retriever.invoke("What is Adeovirus infection?")
for i, doc in enumerate(docs):
    print(f"\n--- Document {i} ---")
    print(doc.page_content[:300])


--- Document 0 ---
addition to the virus, there are usually two other types of
living creatures involved in the cycle leading to human
disease. When large quantities of virus are present in an
arthropod (often a tick or mosquito), the viruses are
passed to a bird or small mammal when the arthropod
attempts to feed on 

--- Document 1 ---
Arbovirus encephalitis
Definition
Encephalitis is a serious inflammation of the brain.
Arbovirus encephalitis is caused by a virus from the
Arbovirus group. The term arbovirus stands for Arthro-
pod-borne virus because these viruses are passed to
humans by members of the phylum Arthropoda (which
inc

--- Document 2 ---
types due to infections they had as children.
In one mode of adenovirus infection (called lytic
infection because it destroys large numbers of cells), ade-
noviruses kill healthy cells and replicate up to one mil-
lion new viruses per cell killed (of which 1-5% are infec-
tious). People with this ki


In [ ]:
pip show langchain

Name: langchain
Version: 1.2.15
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: c:\users\venkatesh\anaconda3\envs\medibot\lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [40]:
pip show langchain pinecone

Name: langchain
Version: 1.2.15
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: c:\users\venkatesh\anaconda3\envs\medibot\lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: pinecone
Version: 7.3.0
Summary: Pinecone client and SDK
Home-page: https://www.pinecone.io
Author: Pinecone Systems, Inc.
Author-email: support@pinecone.io
License: Apache-2.0
Location: c:\users\venkatesh\anaconda3\envs\medibot\lib\site-packages
Requires: certifi, pinecone-plugin-assistant, pinecone-plugin-interface, python-dateutil, typing-extensions, urllib3
Required-by: langchain-pinecone
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sys
print(sys.executable)

c:\Users\VENKATESH\anaconda3\envs\medibot\python.exe
